# Aprendizado de Máquina — Aula prática 01

## Introdução ao Aprendizado de Máquina

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Esta é a primeira aula prática do curso. Ela acompanha as notas da **Aula 01**
([AME] cap. 1; [ISLP] cap. 1–2) e tem um fio condutor único:

> **aqui nós conhecemos a função de regressão $r(x)$.**

Isso é um luxo que nunca teremos com dados reais, e é justamente por isso que
começamos assim. Tudo o que a Aula 01 define no papel — risco, erro irredutível,
viés, variância — é uma quantidade que *podemos calcular* quando somos nós que
geramos os dados. Vamos calcular cada uma delas e conferir, numericamente, os dois
resultados teóricos da aula: a Proposição que diz que $r$ minimiza o risco
quadrático, e o Teorema da decomposição viés–variância.

Só no final abrimos um conjunto de dados real — e a primeira coisa que vamos notar
é que nada daquilo é mais observável.

### Objetivos

Ao final deste notebook você deve ser capaz de:

- simular um problema de aprendizado supervisionado e visualizar $r(x)$;
- verificar numericamente que $r(x)=\mathbb{E}[Y\mid X=x]$ minimiza o risco quadrático;
- distinguir, no código, o que é **predição** e o que é **inferência**;
- medir o **otimismo** do erro de treino como estimador do risco;
- reproduzir a curva em "U" do erro de teste e o painel sub/superajuste;
- estimar **viés²**, **variância** e **erro irredutível** por simulação e conferir
  que somam o erro quadrático médio;
- reconhecer o papel de um *tuning parameter*.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

Além dessas, cada notebook declara em um segundo bloco os objetos **novos** —
aqueles que aparecem pela primeira vez aqui. Neste notebook são os módulos do
`scikit-learn` que montam e avaliam um modelo, e o `statsmodels`, que usaremos uma
única vez, na Seção 5, para contrastar as duas culturas de modelagem.

In [ ]:
import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error

import statsmodels.api as sm

Ajustes polinomiais de grau alto sobre poucos pontos produzem avisos de
convergência do `scikit-learn` que não afetam o resultado. Vamos silenciá-los para
não poluir a saída — mas note que, em geral, **ler os avisos é parte do trabalho**.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

---
## 2. O ambiente de trabalho

Uma passada rápida pelas três bibliotecas que sustentam o curso inteiro. Se você
já programa em Python, pode ler em diagonal — mas vale parar no `default_rng`, que
é a peça de que mais dependeremos hoje.

### Aleatoriedade reprodutível

Todo experimento deste notebook envolve sorteio. Para que os resultados sejam
**reprodutíveis** — mesmos números a cada execução, na sua máquina e na minha —
criamos um *gerador* com semente fixa e passamos esse gerador adiante, em vez de
usar as funções globais `np.random.*`.

In [ ]:
rng = np.random.default_rng(0)     # 0 e a semente: troque e tudo muda

print("3 normais padrao:", rng.normal(size=3))
print("3 uniformes em [-3,3]:", rng.uniform(-3, 3, size=3))

Repare que uma segunda chamada devolve números **diferentes**: o gerador avança de
estado. Reproduzir exige recriar o gerador com a mesma semente.

In [ ]:
print("segunda chamada :", rng.normal(size=3))
print("gerador recriado:", np.random.default_rng(0).normal(size=3))

### Arrays e vetorização

Um `ndarray` do `numpy` é um bloco homogêneo de números com uma forma (`shape`).
Operações se aplicam elemento a elemento, sem laço explícito — isso se chama
**vetorização**, e é o que torna o `numpy` rápido.

In [ ]:
x = np.linspace(-3, 3, 7)          # 7 pontos igualmente espacados
print("x        =", x)
print("shape    =", x.shape)
print("x**2     =", x**2)          # elemento a elemento, sem laco
print("media    =", x.mean(), " | desvio:", x.std())

Duas convenções que vão reaparecer o curso todo: o `scikit-learn` espera a matriz
de covariáveis com **duas dimensões**, `(n, d)`, mesmo quando $d=1$. Converter um
vetor para coluna é o papel do `reshape(-1, 1)`.

In [ ]:
print("vetor          :", x.shape)
print("coluna (n, 1)  :", x.reshape(-1, 1).shape)

### Gráficos

Usamos a interface orientada a objeto do `matplotlib`: `subplots()` devolve a
figura e o(s) eixo(s), e todo o desenho acontece em métodos do eixo (`ax.plot`,
`ax.set_xlabel`, ...). É o estilo do [ISLP] e o que permite compor painéis com
vários gráficos.

In [ ]:
fig, ax = subplots(figsize=(6, 4))
ax.plot(x, x**2, "o-", label="$x^2$")
ax.plot(x, x**3 / 3, "s--", label="$x^3/3$")
ax.set_xlabel("$x$")
ax.set_ylabel("valor")
ax.legend();

### `pandas`

Um `DataFrame` é uma tabela com colunas nomeadas e de tipos possivelmente
diferentes. É o formato em que os dados reais chegam.

In [ ]:
tabela = pd.DataFrame({"x": x, "x2": x**2, "sinal": np.sign(x)})
print(tabela.shape)
tabela.head()

In [ ]:
tabela.describe()

Um exemplo com os três de uma vez: sortear de uma distribuição conhecida, guardar
num `DataFrame` e comparar o que a amostra diz com o que a população dita.

In [ ]:
rng_demo = np.random.default_rng(7)
amostra_normal = pd.DataFrame({"valor": rng_demo.normal(5, 2, size=200)})

fig, ax = subplots(figsize=(6, 4))
ax.hist(amostra_normal["valor"], bins=20, edgecolor="white")
ax.axvline(5, color="r", lw=2, label="média teórica = 5")
ax.axvline(amostra_normal["valor"].mean(), color="k", ls="--", lw=2,
           label=f"média amostral = {amostra_normal['valor'].mean():.4f}")
ax.set_xlabel("valor")
ax.set_ylabel("frequência")
ax.legend();

In [ ]:
media = amostra_normal["valor"].mean()
erro_padrao = 2 / np.sqrt(200)          # dp da populacao dividido por raiz de n

print(f"media amostral : {media:.4f}   (teorica: 5)")
print(f"desvio amostral: {amostra_normal['valor'].std():.4f}   (teorico: 2)")
print(f"erro-padrao da media: {erro_padrao:.4f}")
print(f"distancia ate a media teorica, em erros-padrao: {abs(media - 5) / erro_padrao:.2f}")

A média amostral deu $4{,}7361$, e não $5$. Isso **não** é um problema do sorteio:
a média de 200 observações tem erro-padrão $2/\sqrt{200} = 0{,}1414$, e estamos a
$1{,}87$ erros-padrão do valor teórico — dentro do que se espera em cerca de 94%
das vezes.

Vale fixar a leitura desde já, porque ela vai reaparecer o curso inteiro:
**"bater" nunca significa "dar exatamente igual"**; significa "cair dentro da
incerteza de amostragem". Toda vez que compararmos um número medido com um número
teórico, a pergunta certa é *quantos erros-padrão de distância*, não *quantas
casas decimais coincidem*.

---
## 3. O problema supervisionado, com as cartas na mesa

Vamos construir a nossa **população**. Fixamos:

$$X \sim \mathrm{Uniforme}(-3,\,3), \qquad
  Y = r(X) + \varepsilon, \qquad
  \varepsilon \sim N(0,\sigma^2)\ \text{independente de } X,$$

com a função de regressão

$$r(x) = \operatorname{sen}(1{,}5x) + 0{,}3x .$$

Essa é a distribuição conjunta $P$ das notas de aula. A diferença é que aqui **nós
a escolhemos** — e portanto conhecemos $r$, conhecemos $\sigma^2$, e podemos
sortear quantas amostras quisermos.

In [ ]:
def r(x):
    """A funcao de regressao verdadeira. So existe porque nos a inventamos."""
    return np.sin(1.5 * x) + 0.3 * x

SIGMA = 0.7            # desvio-padrao do ruido
A, B = -3.0, 3.0       # X ~ Uniforme(A, B)


def amostra(n, rng):
    """Sorteia n pares (x, y) da populacao."""
    x = rng.uniform(A, B, size=n)
    y = r(x) + rng.normal(0, SIGMA, size=n)
    return x, y

Uma observação sobre a notação: $\sigma^2$ é a variância condicional
$\operatorname{Var}(Y \mid X = x)$, que neste exemplo **não depende de $x$**
(homocedasticidade). As notas escrevem $\sigma^2(x_0)$ justamente porque, em geral,
ela pode variar com $x$.

In [ ]:
rng = np.random.default_rng(0)
x_obs, y_obs = amostra(50, rng)

grade = np.linspace(A, B, 400)

fig, ax = subplots(figsize=(8, 5))
ax.scatter(x_obs, y_obs, s=25, c="k", alpha=0.6, label="amostra observada")
ax.plot(grade, r(grade), "r-", lw=2.5, label="$r(x)$ — o que queremos estimar")
ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
ax.set_title("A população e uma amostra de tamanho 50")
ax.legend();

**A curva vermelha é o alvo.** Os pontos pretos são tudo o que um algoritmo de
aprendizado enxerga. "Aprender", neste curso, é o exercício de reconstruir a curva
vermelha a partir dos pontos pretos.

E note o que a figura já diz: os pontos **não caem** sobre a curva. A dispersão
vertical em torno dela é o $\varepsilon$ — ruído que nenhum método, por melhor
que seja, poderá explicar. Guarde isso: é o **erro irredutível**, e daqui a pouco
vamos medi-lo.

A curva vermelha é o alvo — mas o quanto ela é **visível** a partir dos pontos
depende inteiramente de $\sigma$. Vamos refazer a figura com o ruído dez vezes
menor e três vezes maior que o nosso.

In [ ]:
fig, axes = subplots(1, 3, figsize=(15, 4.2), sharey=True)

for ax, sigma in zip(axes, [0.2, SIGMA, 2.0]):
    rng_s = np.random.default_rng(0)                  # mesma semente: mesmos x
    x_s = rng_s.uniform(A, B, size=50)
    y_s = r(x_s) + rng_s.normal(0, sigma, size=50)

    ax.scatter(x_s, y_s, s=25, c="k", alpha=0.6)
    ax.plot(grade, r(grade), "r-", lw=2.5)
    ax.set_xlabel("$x$")
    ax.set_title(rf"$\sigma = {sigma}$   ($\sigma^2 = {sigma**2:.2f}$)")

axes[0].set_ylabel("$y$")
axes[0].set_ylim(-6, 6);

In [ ]:
print("sigma   Var(y)   sigma^2   fracao de Var(y) que r explica")
for sigma in (0.2, SIGMA, 2.0):
    rng_s = np.random.default_rng(0)
    x_s = rng_s.uniform(A, B, size=50)
    y_s = r(x_s) + rng_s.normal(0, sigma, size=50)
    print(f" {sigma:4.1f}   {y_s.var():6.4f}   {sigma**2:7.4f}   "
          f"{1 - sigma**2 / y_s.var():.4f}")

Com $\sigma = 0{,}2$ os pontos praticamente desenham a curva: **95,3%** da
variação de $y$ vem de $r(X)$, e sobra pouco para o ruído. Com $\sigma = 2$ a
curva some dentro da nuvem — apenas **23,0%** da variação é atribuível a $r$, e
nenhum método vai recuperar o que não está lá.

O ponto não é sobre método nenhum. A dificuldade de um problema de aprendizado é
uma propriedade da **população**, não do algoritmo: a mesma $r$, a mesma
quantidade de dados e o mesmo estimador produzem resultados incomparáveis conforme
$\sigma$. É por isso que comparar acurácias entre problemas diferentes não
significa nada, e é a razão de o $\sigma^2$ aparecer como parcela separada na
decomposição da Seção 8.

---
## 4. Por que $r(x) = \mathbb{E}[Y \mid X = x]$?

A Proposição 1 das notas afirma que, entre **todas** as funções
$g:\mathbb{R}^d \to \mathbb{R}$, a função de regressão minimiza o erro quadrático
esperado:

$$r = \arg\min_{g} \ \mathbb{E}\big[(Y - g(X))^2\big].$$

Esse é o tipo de afirmação que se demonstra em três linhas e se acredita em zero.
Vamos conferir. Sorteamos uma amostra enorme (200 mil pontos) e calculamos o risco
empírico de vários candidatos $g$ — incluindo o próprio $r$, e incluindo algumas
distorções deliberadas dele.

In [ ]:
rng = np.random.default_rng(10)
x_big, y_big = amostra(200_000, rng)

reta = skl.LinearRegression().fit(x_big.reshape(-1, 1), y_big)

candidatos = {
    "r(x)  — a função de regressão": r(x_big),
    "r(x) + 0.5  (deslocada)":       r(x_big) + 0.5,
    "0.8 * r(x)  (achatada)":        0.8 * r(x_big),
    "melhor reta (MQO)":             reta.predict(x_big.reshape(-1, 1)),
    "constante E[Y]":                np.full_like(x_big, y_big.mean()),
}

print(f"{'candidato g':38s}  risco estimado")
print("-" * 56)
for nome, pred in candidatos.items():
    print(f"{nome:38s}  {np.mean((y_big - pred)**2):.4f}")

Nenhum candidato bate $r$ — e não por pouco. Mais interessante é *quanto* vale o
risco de $r$: compare-o com $\sigma^2$.

In [ ]:
risco_r = np.mean((y_big - r(x_big))**2)

print(f"risco de r(x)      R(r) = {risco_r:.4f}")
print(f"erro irredutivel  sigma^2 = {SIGMA**2:.4f}")
print(f"diferenca                = {abs(risco_r - SIGMA**2):.4f}")

São o mesmo número (a menos do erro de Monte Carlo). Isso é a equação de
decomposição do risco das notas em ação:

$$R(g) = \underbrace{\mathbb{E}\big[(g(X)-r(X))^2\big]}_{\text{excesso de risco}}
       + \underbrace{\mathbb{E}\big[\operatorname{Var}(Y\mid X)\big]}_{\text{erro irredutível}}.$$

Para $g = r$ o primeiro termo zera e sobra só o segundo. **Mesmo conhecendo $r$
perfeitamente, o risco não vai a zero.** Esse piso, $\sigma^2 = 0{,}49$, vai
reaparecer em todos os gráficos do resto do notebook — é o melhor que qualquer
método pode aspirar.

---
## 5. Predição × inferência: as duas culturas

As notas (§2) distinguem duas razões para estimar $r$:

- **predição** — quero um bom palpite para o $Y$ de uma nova observação. O modelo
  pode ser uma caixa-preta; o que importa é o acerto.
- **inferência** — quero *entender* a relação entre $X$ e $Y$. Quais covariáveis
  importam? Com que sinal? A interpretabilidade é o produto.

Breiman (2001) chamou essas posturas de as duas *culturas* da modelagem. A melhor
forma de sentir a diferença é rodar as duas sobre os **mesmos dados** e olhar o que
cada uma imprime na tela.

Para isso trocamos de exemplo: um modelo linear com 6 covariáveis, das quais só 3
têm efeito de verdade.

In [ ]:
rng = np.random.default_rng(2)

n, d = 200, 6
beta_verdadeiro = np.array([2.0, -1.5, 0.0, 0.0, 0.8, 0.0])

X = rng.normal(size=(n, d))
y = X @ beta_verdadeiro + rng.normal(0, 1.0, size=n)

nomes = [f"x{j+1}" for j in range(d)]
print("beta verdadeiro:", dict(zip(nomes, beta_verdadeiro)))

### Cultura 1 — predição (`scikit-learn`)

A interface do `scikit-learn` é deliberadamente pobre em estatística: ela oferece
`fit`, `predict` e uma métrica. Isso não é limitação, é postura — a pergunta que
ela responde é *"quão bem eu acerto?"*.

In [ ]:
X_tr, X_te, y_tr, y_te = skm.train_test_split(X, y, test_size=0.3, random_state=0)

modelo = skl.LinearRegression().fit(X_tr, y_tr)

print("EQM no teste:", mean_squared_error(y_te, modelo.predict(X_te)).round(4))
print("coeficientes:", modelo.coef_.round(3))

Os coeficientes estimados estão lá, mas repare: **nada** na saída lhe diz se o
$0{,}0\!\ldots$ estimado para `x3` é "zero de verdade" ou ruído. O
`scikit-learn` não tem opinião sobre isso, porque não é a pergunta dele.

### Cultura 2 — inferência (`statsmodels`)

O `statsmodels` responde à outra pergunta. Mesmos dados, mesmo modelo linear.

In [ ]:
X_df = pd.DataFrame(X, columns=nomes)
ajuste = sm.OLS(y, sm.add_constant(X_df)).fit()

print(ajuste.summary().tables[1])

Agora cada coeficiente vem com erro-padrão, estatística $t$, $p$-valor e intervalo
de confiança. Confira contra o `beta_verdadeiro`: `x1`, `x2` e `x5` aparecem com
$p$-valores minúsculos, e `x3`, `x4`, `x6` com $p$-valores grandes e intervalos que
contêm o zero. A inferência **acertou quais covariáveis importam** — e essa era a
pergunta.

Dois avisos, porém:

1. Ganhamos os $p$-valores porque *assumimos* o modelo linear com erros normais.
   Toda a tabela acima depende dessa suposição; se ela for falsa, os números
   continuam sendo impressos, e continuam errados.
2. Nada garante que o modelo com melhor $p$-valor seja o que melhor prediz. São
   critérios diferentes.

> **Para discussão.** As notas sugerem o contraste: previsão de demanda de energia
> (predição pura) *versus* avaliar o efeito de um medicamento (inferência). Em qual
> desses um modelo caixa-preta altamente preciso pode ser **inaceitável**? E existe
> algum caso em que um modelo interpretável, porém pior, seja a escolha certa?

Guarde o `beta_verdadeiro` esparso deste exemplo: ele volta na **Aula prática 02**,
quando perguntarmos se o Lasso consegue descobrir sozinho quais coeficientes são
nulos — sem precisar assumir normalidade nenhuma.

---
## 6. Risco empírico e o otimismo do erro de treino

Voltamos à população unidimensional da Seção 3. O risco

$$R(g) = \mathbb{E}[L(Y, g(X))]$$

é uma esperança sob $P$, que na vida real não conhecemos. O que temos é o **risco
empírico**, a média das perdas sobre uma amostra:

$$\widehat{R}(g) = \frac1n \sum_{i=1}^n L\big(Y_i, g(X_i)\big).$$

A pergunta desta seção é: pode-se calcular $\widehat{R}$ **nos mesmos dados usados
para ajustar** $g$? As notas dizem que não, e chamam esse erro de *otimista*. Vamos
medir o tamanho do otimismo.

Primeiro, o objeto que usaremos daqui até o fim: um ajuste polinomial de grau $g$,
montado como um `Pipeline`.

In [ ]:
def modelo_poly(grau):
    """Regressao polinomial de grau `grau`, como um Pipeline de 3 passos."""
    return Pipeline([
        ("poly",   PolynomialFeatures(degree=grau, include_bias=False)),
        ("escala", StandardScaler()),
        ("mqo",    skl.LinearRegression()),
    ])

O `Pipeline` encadeia transformações e o modelo final em um único objeto com `fit`
e `predict`. Aqui ele constrói as potências $x, x^2, \dots, x^{\text{grau}}$,
padroniza cada uma (potências altas produzem colunas em escalas absurdamente
diferentes) e ajusta mínimos quadrados. Na **Aula 07** veremos que o `Pipeline` é
muito mais que conveniência: é o que impede vazamento de dados.

Agora o experimento: uma amostra de treino de 50 pontos, uma de teste com 5 000, e
um polinômio de grau 15.

In [ ]:
N_TR = 50

rng = np.random.default_rng(0)
x_tr, y_tr = amostra(N_TR, rng)
x_te, y_te = amostra(5_000, rng)

ajuste15 = modelo_poly(15).fit(x_tr.reshape(-1, 1), y_tr)

err_tr = mean_squared_error(y_tr, ajuste15.predict(x_tr.reshape(-1, 1)))
err_te = mean_squared_error(y_te, ajuste15.predict(x_te.reshape(-1, 1)))

print(f"erro de TREINO (grau 15) : {err_tr:.3f}")
print(f"erro de TESTE  (grau 15) : {err_te:.3f}")
print(f"erro irredutivel sigma^2 : {SIGMA**2:.3f}")

Olhe o erro de treino: **0,296**, contra um erro irredutível de **0,49**. O modelo
está reportando um erro *menor que o piso teórico do problema*. Isso é
impossível para o risco verdadeiro — e é exatamente o sintoma. O polinômio de grau
15 não aprendeu $r$; ele decorou os 50 pontos, ruído incluído. No teste, o erro é
mais que o dobro.

**Nunca reporte erro de treino como medida de desempenho.**

---
## 7. Sub e superajuste: a curva em "U"

Repetimos o ajuste para todos os graus de 1 a 15, medindo os dois erros.

In [ ]:
graus = np.arange(1, 16)

erros_tr, erros_te = [], []
for g in graus:
    m = modelo_poly(g).fit(x_tr.reshape(-1, 1), y_tr)
    erros_tr.append(mean_squared_error(y_tr, m.predict(x_tr.reshape(-1, 1))))
    erros_te.append(mean_squared_error(y_te, m.predict(x_te.reshape(-1, 1))))

erros_tr, erros_te = np.array(erros_tr), np.array(erros_te)

resumo = pd.DataFrame({"grau": graus,
                       "erro_treino": erros_tr.round(3),
                       "erro_teste": erros_te.round(3)}).set_index("grau")
resumo

In [ ]:
fig, ax = subplots(figsize=(8, 5))
ax.plot(graus, erros_tr, "o-", label="erro de treino")
ax.plot(graus, erros_te, "s-", label="erro de teste")
ax.axhline(SIGMA**2, ls="--", c="gray", label=r"erro irredutível $\sigma^2$")
ax.axvline(graus[erros_te.argmin()], ls=":", c="green",
           label=f"melhor grau = {graus[erros_te.argmin()]}")
ax.set_xlabel(r"grau do polinômio (flexibilidade $\rightarrow$)")
ax.set_ylabel("EQM")
ax.set_xticks(graus)
ax.legend();

Os dois comportamentos que as notas anunciam, lado a lado:

- o **erro de treino** cai monotonicamente. Sempre. Aumentar a flexibilidade nunca
  piora o ajuste aos dados que você já tem — e é por isso que ele não serve como
  critério de escolha;
- o **erro de teste** tem forma de **U**: cai enquanto o modelo ainda está rígido
  demais (subajuste), atinge um mínimo, e volta a subir quando o modelo passa a
  ajustar ruído (superajuste).

Repare também onde a curva azul cruza a linha cinza: a partir do grau 7 o erro de
treino fica **abaixo do erro irredutível**. Da esquerda para a direita desse ponto,
o modelo está literalmente explicando ruído.

Vamos ver o que cada regime significa graficamente.

In [ ]:
fig, axes = subplots(1, 3, figsize=(15, 4.2))
grade_col = grade.reshape(-1, 1)

for ax, g in zip(axes, [1, 5, 15]):
    m = modelo_poly(g).fit(x_tr.reshape(-1, 1), y_tr)
    ax.scatter(x_tr, y_tr, s=22, c="k", alpha=0.6, label="amostra de treino")
    ax.plot(grade, r(grade), "r-", lw=2, label="$r(x)$ verdadeira")
    ax.plot(grade, m.predict(grade_col), "b-", lw=2, label=f"ajuste grau {g}")
    ax.set_ylim(-3.5, 3.5)
    ax.set_xlabel("$x$")
    ax.set_title(f"grau {g}  |  treino {erros_tr[g-1]:.2f}  ·  teste {erros_te[g-1]:.2f}")
    ax.legend(fontsize=8, loc="upper left")

- **Grau 1 (subajuste).** A reta não tem como capturar um seno. Erra em toda parte,
  e erraria do mesmo jeito com mil vezes mais dados. O problema é a classe
  $\mathcal{H}$, não a amostra. Modelo *enviesado*.
- **Grau 5.** Praticamente sobre a curva vermelha. Erro de teste 0,51, contra o piso
  de 0,49 — quase o ótimo possível.
- **Grau 15 (superajuste).** As oscilações nas bordas são a assinatura clássica.
  O modelo passa perto de quase todos os pontos pretos e, ao fazer isso, se afasta
  do vermelho. Modelo *variável*.

A curva em U acima veio de **uma** amostra de treino de 50 pontos. O que muda com
dez vezes mais dados? A resposta tem uma parte previsível e uma parte que costuma
surpreender.

In [ ]:
comparacao = {}

for n_treino in (50, 500):
    rng_g = np.random.default_rng(0)
    x_g, y_g = amostra(n_treino, rng_g)
    x_gte, y_gte = amostra(5_000, rng_g)

    tr_g, te_g = [], []
    for g in graus:
        m = modelo_poly(g).fit(x_g.reshape(-1, 1), y_g)
        tr_g.append(mean_squared_error(y_g, m.predict(x_g.reshape(-1, 1))))
        te_g.append(mean_squared_error(y_gte, m.predict(x_gte.reshape(-1, 1))))
    comparacao[n_treino] = (np.array(tr_g), np.array(te_g))

    melhor = graus[np.array(te_g).argmin()]
    print(f"N_TR = {n_treino:3d}:  melhor grau {melhor:2d}   "
          f"teste no minimo {min(te_g):.4f}   teste no grau 15 {te_g[-1]:.4f}   "
          f"(razao {te_g[-1] / min(te_g):.2f}x)")

In [ ]:
fig, ax = subplots(figsize=(8, 5))
for n_treino, (tr_g, te_g) in comparacao.items():
    ax.plot(graus, te_g, "o-", label=f"erro de teste, $n = {n_treino}$")
ax.axhline(SIGMA**2, ls="--", c="gray", label=r"$\sigma^2$")
ax.set_xlabel(r"grau do polinômio (flexibilidade $\rightarrow$)")
ax.set_ylabel("EQM de teste")
ax.set_xticks(graus)
ax.legend();

**O grau que minimiza o erro de teste não se move: continua sendo 5.** Isso
contraria a intuição de que "mais dados permitem modelos mais flexíveis" — e a
intuição não está errada, está incompleta.

O que ela descreve é o caso em que a classe $\mathcal{H}$ ainda não contém $r$.
Aqui não é o caso: um polinômio de grau 5 já aproxima
$\operatorname{sen}(1{,}5x) + 0{,}3x$ quase exatamente no intervalo $[-3,3]$, e
graus maiores não têm nada de novo a oferecer. Mais dados não criam estrutura que
a função-alvo não tem.

**O que muda é a profundidade do U.** Com $n=50$, o grau 15 custa 1,48 vez o erro
do grau 5; com $n=500$, custa 1,01 vez — o U vira praticamente uma linha reta. Em
outras palavras: com poucos dados, escolher errado é caro; com muitos, quase não
importa.

É essa a versão precisa da intuição. Mais dados **não** deslocam o ótimo para a
direita quando a classe já contém a verdade; eles **aplanam a penalidade** por
errar a escolha. E note a consequência prática, que a Aula 03 vai explorar: é
exatamente quando você tem poucos dados — e portanto mais precisa acertar o
hiperparâmetro — que estimar o risco é mais difícil.

---
## 8. A decomposição viés–variância

A curva da seção anterior veio de **uma** amostra de treino. Se sorteássemos outra,
os números mudariam — e essa variação é, ela própria, o objeto de estudo desta
seção.

O Teorema 1 das notas diz que, em um ponto $x_0$,

$$\underbrace{\mathbb{E}_{\mathcal{D}}\big[(Y_0-\hat r(x_0))^2\big]}_{\text{EQM}}
= \underbrace{\big(\mathbb{E}_{\mathcal{D}}[\hat r(x_0)]-r(x_0)\big)^2}_{\text{viés}^2}
+ \underbrace{\operatorname{Var}_{\mathcal{D}}\big(\hat r(x_0)\big)}_{\text{variância}}
+ \underbrace{\sigma^2}_{\text{irredutível}}.$$

As esperanças são **sobre o sorteio da amostra de treino** $\mathcal{D}$. Ou seja:
para estimar esses três termos precisamos de muitas amostras de treino. Na vida
real temos uma só — mas aqui somos donos da população.

O plano: para cada grau, sortear $B=500$ amostras de treino independentes, ajustar
o modelo em cada uma, e olhar a **distribuição** das 500 predições em cada ponto de
uma grade.

In [ ]:
B_SIM = 500
grade_bv = np.linspace(A, B, 200).reshape(-1, 1)
r_grade = r(grade_bv.ravel())

rng_bv = np.random.default_rng(1)
vies2, variancia, eqm = [], [], []

for g in graus:
    preds = np.empty((B_SIM, grade_bv.shape[0]))
    for b in range(B_SIM):
        xb, yb = amostra(N_TR, rng_bv)
        preds[b] = modelo_poly(g).fit(xb.reshape(-1, 1), yb).predict(grade_bv)

    media_preds = preds.mean(axis=0)                       # E_D[ r_hat(x0) ]
    vies2.append(np.mean((media_preds - r_grade) ** 2))
    variancia.append(np.mean(preds.var(axis=0)))
    eqm.append(np.mean((preds - r_grade) ** 2) + SIGMA**2)

vies2, variancia, eqm = np.array(vies2), np.array(variancia), np.array(eqm)
print("simulacao concluida:", B_SIM, "amostras de treino por grau")

Cada linha de `preds` é uma curva ajustada a partir de uma amostra diferente.
A **média** dessas curvas, comparada com $r$, dá o viés; a **dispersão** delas em
torno da própria média dá a variância. Vamos conferir se os três termos somam o EQM.

In [ ]:
tabela_bv = pd.DataFrame({
    "grau": graus,
    "vies2": vies2,
    "variancia": variancia,
    "sigma2": SIGMA**2,
    "soma": vies2 + variancia + SIGMA**2,
    "EQM": eqm,
}).set_index("grau")
tabela_bv["|soma - EQM|"] = (tabela_bv["soma"] - tabela_bv["EQM"]).abs()

print("maior discrepancia da identidade:", tabela_bv["|soma - EQM|"].max())
tabela_bv.round(4)

A maior discrepância é da ordem de $10^{-12}$ — ou seja, **zero**, a menos de erro
de arredondamento de ponto flutuante. O Teorema não é uma aproximação: é uma
identidade, e ela fecha exatamente.

Agora a figura. Restringimos aos graus 1–12: além disso a variância cresce tanto
que o gráfico deixa de ser legível.

In [ ]:
corte = 12
fig, ax = subplots(figsize=(8.5, 5.5))
ax.plot(graus[:corte], vies2[:corte], "o-", label=r"viés$^2$")
ax.plot(graus[:corte], variancia[:corte], "s-", label="variância")
ax.axhline(SIGMA**2, ls="--", c="gray", label=r"$\sigma^2$ (irredutível)")
ax.plot(graus[:corte], eqm[:corte], "k^-", lw=2.5, label="EQM = soma dos três")
ax.axvline(graus[eqm.argmin()], ls=":", c="green")
ax.set_yscale("log")
ax.set_xlabel(r"grau do polinômio (flexibilidade $\rightarrow$)")
ax.set_ylabel("contribuição para o EQM (escala log)")
ax.set_xticks(graus[:corte])
ax.legend();

Esta é **a** figura da Aula 01:

- o **viés²** despenca do grau 1 ao 5 — sair da reta para o polinômio cúbico
  resolve quase todo o problema de aproximação, e a partir do grau 5 a classe
  $\mathcal{H}$ já contém algo praticamente igual a $r$;
- a **variância** sobe monotonicamente, e depois do grau 8 explode: mais parâmetros
  significam um ajuste mais sensível a qual amostra de 50 pontos você sorteou;
- o **$\sigma^2$** é uma linha reta. Não depende do modelo. Nenhuma escolha de grau
  mexe nele;
- o **EQM** é a soma dos três, e por isso tem o "U" — o mesmo "U" da Seção 7, agora
  decomposto em suas causas. O mínimo está no grau 5, onde o ganho marginal em viés
  passa a ser menor que a perda marginal em variância.

Selecionar um modelo é **escolher um ponto nesse balanço**. Com $n$ finito não dá
para zerar os dois.

> **Uma honestidade sobre a figura.** O viés² parece subir a partir do grau 9. Isso
> é artefato de simulação, não teoria: estimar $\mathbb{E}_{\mathcal{D}}[\hat r(x_0)]$
> quando $\operatorname{Var}(\hat r(x_0))$ é da ordem de centenas exigiria muito
> mais que $B=500$ réplicas. Aumente `B_SIM` e veja a curva azul achatar.

A decomposição acima usou amostras de treino de 50 pontos. Repetindo tudo com
$n=200$, dá para ver **qual das três parcelas** responde ao tamanho da amostra.

In [ ]:
N_TR_G = 200
rng_g2 = np.random.default_rng(1)
vies2_g, variancia_g, eqm_g = [], [], []

for g in graus:
    preds_g = np.empty((B_SIM, grade_bv.shape[0]))
    for b in range(B_SIM):
        xb, yb = amostra(N_TR_G, rng_g2)
        preds_g[b] = modelo_poly(g).fit(xb.reshape(-1, 1), yb).predict(grade_bv)

    vies2_g.append(np.mean((preds_g.mean(axis=0) - r_grade) ** 2))
    variancia_g.append(np.mean(preds_g.var(axis=0)))
    eqm_g.append(np.mean((preds_g - r_grade) ** 2) + SIGMA**2)

vies2_g, variancia_g, eqm_g = map(np.array, (vies2_g, variancia_g, eqm_g))

pd.DataFrame({
    "grau": graus,
    "vies2 (n=50)":  vies2.round(4),    "vies2 (n=200)":  vies2_g.round(4),
    "var (n=50)":    variancia.round(4), "var (n=200)":    variancia_g.round(4),
    "EQM (n=50)":    eqm.round(4),       "EQM (n=200)":    eqm_g.round(4),
}).set_index("grau").loc[[1, 5, 10, 15]]

In [ ]:
print(f"melhor grau com n=50 : {graus[eqm.argmin()]}   (EQM {eqm.min():.4f})")
print(f"melhor grau com n=200: {graus[eqm_g.argmin()]}   (EQM {eqm_g.min():.4f})")
print(f"\nvariancia no grau 15: {variancia[14]:.4f} -> {variancia_g[14]:.4f}"
      f"   (fator de {variancia[14] / variancia_g[14]:.0f})")
print(f"vies2     no grau 15: {vies2[14]:.4f} -> {vies2_g[14]:.4f}")

A tabela separa as duas parcelas de forma limpa:

- o **viés²** praticamente não muda onde ele é grande e bem estimado: no grau 1
  vai de $0{,}4797$ para $0{,}4796$, e no grau 5, de $0{,}0018$ para $0{,}0017$.
  Faz sentido: viés é a distância entre a *melhor* função da classe e $r$, e a
  classe não mudou. **Nenhuma quantidade de dados corrige viés de classe;**
- a **variância** desaba. No grau 15 ela cai de $6\,027$ para $0{,}139$ — um fator
  de mais de **43 mil**. Variância é sensibilidade a *qual* amostra você sorteou, e
  quadruplicar a amostra a reduz.

E o mínimo do EQM continua no grau 5, pelo mesmo motivo do exemplo anterior: o
viés já era desprezível a partir dali, então não havia o que ganhar indo além.

Nos graus 10 e 15 o viés² *parece* cair muito ($0{,}0047 \to 0{,}0000$ e
$0{,}2707 \to 0{,}0001$), e aqui a explicação é outra: com $n=50$ aqueles
números não eram viés, eram **ruído de simulação** — é exatamente a ressalva da
caixa acima. Estimar $\mathbb{E}_{\mathcal{D}}[\hat r(x_0)]$ com $B=500$ quando a
variância é da ordem de $6\,000$ deixa um resíduo que se confunde com viés. Com
$n=200$ a variância cai para $0{,}14$, a média das 500 curvas fica bem estimada, e
o viés² aparece pelo que sempre foi: praticamente zero. O artefato sumiu porque a
sua causa sumiu.

Junte as duas leituras e você tem o resumo da Aula 01: **mais dados compram
variância, não viés.** Se o seu modelo está errado por ser rígido demais, coletar
mais dados não resolve — é preciso trocar a classe. Se está errado por ser
instável, mais dados resolvem sozinhos.

---
## 9. *Tuning parameters*

O grau do polinômio é o primeiro **hiperparâmetro** (ou *tuning parameter*) do
curso: um número que controla a flexibilidade da classe $\mathcal{H}$ e que
**não** é estimado junto com o ajuste — é escolhido por fora.

| Método | Hiperparâmetro | Efeito de aumentá-lo |
|---|---|---|
| Regressão polinomial | grau | mais flexível (↑ variância) |
| Ridge / Lasso | $\lambda$ | mais rígido (↑ viés) |
| $k$ vizinhos (KNN) | $k$ | mais rígido (↑ viés) |
| Árvore | profundidade / $\alpha$ de poda | depende do parâmetro |

Todos serão escolhidos da mesma forma: estimando o risco para vários valores e
tomando o melhor.

E aqui está o problema que fecha a aula. Na Seção 7 nós escolhemos o grau 5 —
**olhando o erro de teste**. Isso é trapaça. Aquele conjunto de teste era a nossa
única estimativa honesta do risco, e nós a gastamos para tomar uma decisão. Se
agora reportarmos "o meu modelo tem EQM 0,509", esse número está contaminado: ele
é o mínimo de 15 tentativas, não o desempenho de um modelo escolhido às cegas.

É uma forma de **vazamento de dados** (*data leakage*), e a solução — validação
cruzada — é o assunto da **Aula 03**. Por ora, a regra:

> hiperparâmetros se escolhem **sem olhar o conjunto de teste**.

---
## 10. Um caso real, em miniatura

Fechamos abrindo dados de verdade: medidas físico-químicas de 21 263
supercondutores, e a temperatura crítica de cada um — a temperatura abaixo da qual
o material perde toda a resistência elétrica. São 81 covariáveis. Voltaremos a
esses dados na Aula prática 02.

In [ ]:
import os

_nome = "superconductivity.csv"
_local = os.path.join("..", "..", "recursos", "dados", _nome)   # repositorio clonado
_url = ("https://raw.githubusercontent.com/HugoCarvalhoUFRJ/ap-maq/"
        "refs/heads/refactoring-baby/recursos/dados/") + _nome  # fallback (ex.: Colab)
_fonte = _local if os.path.exists(_local) else _url

df = pd.read_csv(_fonte)
print("dimensoes:", df.shape)
df.head()

In [ ]:
X_sc = df.drop(columns="critical_temp").values
y_sc = df["critical_temp"].values

X_tr, X_te, y_tr_sc, y_te_sc = skm.train_test_split(
    X_sc, y_sc, test_size=0.3, random_state=0)

modelo = skl.LinearRegression().fit(X_tr, y_tr_sc)

print(f"EQM no treino : {mean_squared_error(y_tr_sc, modelo.predict(X_tr)):8.3f}")
print(f"EQM no teste  : {mean_squared_error(y_te_sc, modelo.predict(X_te)):8.3f}")
print(f"variancia de Y: {y_sc.var():8.3f}   <- o EQM do preditor constante")

Este é o esqueleto que repetiremos o curso inteiro, e que já apareceu nas notas:

```python
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
modelo.fit(X_tr, y_tr)                              # ajustar SO no treino
mean_squared_error(y_te, modelo.predict(X_te))      # estimativa de R(g)
```

Compare com tudo o que fizemos antes e note o que **desapareceu**:

- não há curva vermelha. Não sabemos, e nunca saberemos, quanto vale $r(x)$;
- não sabemos $\sigma^2$. Aquele piso que aparecia em todo gráfico — o limite do
  que qualquer método pode atingir — é agora desconhecido. Se o EQM de teste der
  310, não temos como saber se isso é excelente (perto do irredutível) ou péssimo;
- não podemos sortear 500 amostras de treino. Temos **uma**. Viés e variância
  continuam existindo, mas viraram quantidades não observáveis;
- e há uma única estimativa honesta do risco, aquele número do teste, que gastamos
  assim que a usarmos para escolher qualquer coisa.

Nada disso torna a teoria inútil — ao contrário. É porque essas quantidades existem
e obedecem à identidade que verificamos na Seção 8 que sabemos **o que procurar**:
um modelo flexível demais vai superajustar, um rígido demais vai subajustar, e há
um ponto ótimo no meio. A simulação foi o laboratório onde vimos o mecanismo
funcionando com as luzes acesas. O resto do curso é aprender a operá-lo no escuro.

---
## Resumo

| Conceito | Onde apareceu | O que vimos |
|---|---|---|
| $r(x)=\mathbb{E}[Y\mid X=x]$ | §4 | minimiza o risco quadrático entre todas as funções |
| erro irredutível | §4 | $R(r)=\sigma^2=0{,}49$ — o piso do problema |
| predição × inferência | §5 | `sklearn` responde "acerto quanto?"; `statsmodels`, "o que importa?" |
| otimismo do treino | §6 | erro de treino 0,296 < $\sigma^2$: sintoma, não conquista |
| sub/superajuste | §7 | treino cai sempre; teste tem forma de "U" |
| viés–variância | §8 | viés² ↓, variância ↑, soma $+\,\sigma^2$ = EQM (identidade exata) |
| *tuning parameter* | §9 | o grau controla a flexibilidade; escolhê-lo pelo teste é vazamento |

**Leitura recomendada.** [AME] Capítulo 1, especialmente §1.1–1.6.
[ISLP] Capítulos 1 e 2 — em particular §2.1 (*What is statistical learning*) e
§2.2 (*Assessing model accuracy*), cujas Figuras 2.9–2.12 são as versões do livro
das figuras que construímos nas Seções 7 e 8.

**Para praticar.** `Lista teorica 01.pdf` (teórica, com gabarito) e
`Lista prática 01.ipynb` (prática, para completar as lacunas), nesta mesma
pasta.

**A seguir.** A Aula 02 mantém o problema de regressão, mas troca a pergunta: em vez
de escolher a flexibilidade pelo grau de um polinômio, vamos controlá-la
**encolhendo coeficientes** — Ridge e Lasso. E a Aula 03 resolve a dívida que
deixamos na §9: como escolher um hiperparâmetro sem gastar o conjunto de teste.